# Unified FlexDC raw inference + validation notebook v5

This notebook is based on the earlier unified inference notebook structure, but defaults to the current **FlexDC raw** model trained on the W2-dense augmented dataset.

Key defaults:

- starting weights are **equal weights** unless you explicitly choose custom weights;
- P/R focused bounds are **off** by default;
- optimizer still optimizes **Pbar, R, and weights**;
- raw optimization can use either equal raw weighted sum or the paper-form objective approximation computed from raw predictions.


## 1. Environment controls — RUN in Colab or locally

In [ ]:
from pathlib import Path
import os

# Change only these high-level controls.
RUN_ENV = "colab"       # "colab" or "local"
WORKSPACE = Path("/content/workspace") if RUN_ENV == "colab" else Path.cwd().parent

# Repository URLs. Change only if your GitHub remotes differ.
COMDER_REPO_URL = "https://github.com/amenon06/comder-main.git"
FLEXDC_REPO_URL = "https://github.com/peaclab/flexdc-sim.git"
COMDER_BRANCH = "main"
FLEXDC_BRANCH = "main"

# W&B controls.
USE_WANDB = True
WANDB_MODE = "online"      # "online", "offline", or "disabled"
WANDB_PROJECT = "flexdc-condor-inference"
WANDB_ENTITY = "amenon06-boston-university"

print("RUN_ENV:", RUN_ENV)
print("WORKSPACE:", WORKSPACE)
print("USE_WANDB:", USE_WANDB, "WANDB_MODE:", WANDB_MODE)

## 2. Install dependencies — RUN in Colab

In [ ]:
%pip install -q wandb pandas numpy scipy scikit-learn tqdm matplotlib tabulate openpyxl

## 3. Clone repositories — RUN in Colab

Skip this cell locally if the repositories already exist. This restores the same repo-pull behavior as the older notebook.

In [ ]:
if RUN_ENV == "colab":
    !rm -rf "$WORKSPACE"
    !mkdir -p "$WORKSPACE"
    %cd {WORKSPACE}
    !git clone --branch "$COMDER_BRANCH" "$COMDER_REPO_URL" comder-main
    !git clone --branch "$FLEXDC_BRANCH" "$FLEXDC_REPO_URL" flexdc-sim
else:
    print("Local mode: not cloning repos. Make sure paths in the next cell are correct.")

## 4. Define paths — RUN after clone/copy

In [ ]:
import sys
from pathlib import Path

if RUN_ENV == "colab":
    COMDER_ROOT = WORKSPACE / "comder-main"
    FLEXDC_ROOT = WORKSPACE / "flexdc-sim"
else:
    # Edit these if running locally.
    COMDER_ROOT = Path(r"C:/Users/Achuthan Menon/Desktop/Research Work/comder-main")
    FLEXDC_ROOT = Path(r"C:/Users/Achuthan Menon/Desktop/Research Work/flexdc-sim-main")

AM_FLEXDC_ROOT = COMDER_ROOT / "am_flexdc"
TRAIN_DIR = AM_FLEXDC_ROOT / "train"
MODELS_DIR = AM_FLEXDC_ROOT / "models"
DATA_DIR = AM_FLEXDC_ROOT / "data"
RESULTS_DIR = AM_FLEXDC_ROOT / "results" / "unified_eval_runs"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for path in [str(TRAIN_DIR), str(FLEXDC_ROOT / "src")]:
    if path not in sys.path:
        sys.path.insert(0, path)

print("COMDER_ROOT:", COMDER_ROOT)
print("FLEXDC_ROOT:", FLEXDC_ROOT)
print("TRAIN_DIR:", TRAIN_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## 5. Optional Google Drive dataset/model copy — RUN only if needed

Use this if the augmented dataset or model checkpoint is not inside the cloned GitHub repo.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Example copy commands. Edit source paths if needed.
# !mkdir -p "{AM_FLEXDC_ROOT}/data/pilots"
# !mkdir -p "{AM_FLEXDC_ROOT}/models/flexdc_raw"
# !cp -r "/content/drive/MyDrive/path/to/traditionaliso_newqos_pilot_plus_w2_dense_v1_flexdc_configured_objective" "{AM_FLEXDC_ROOT}/data/pilots/"
# !cp "/content/drive/MyDrive/path/to/am_flexdc_raw_newqos_plus_w2dense_v1_wandb_v2_state_dict.pt" "{AM_FLEXDC_ROOT}/models/flexdc_raw/"

## 6. Required-file check — RUN before inference

In [ ]:
from pathlib import Path

required = [
    TRAIN_DIR / "data_center_model.py",
    TRAIN_DIR / "am_unified_predict_one.py",
    TRAIN_DIR / "am_unified_optimize_one_raw_v5.py",
    TRAIN_DIR / "am_unified_end_to_end_eval_raw_report_v5.py",
    TRAIN_DIR / "am_unified_training_utilities.py",
    FLEXDC_ROOT / "src" / "peacsim" / "am_data_extraction_wizard.py",
]

missing = [p for p in required if not p.exists()]
if missing:
    print("Missing required files:")
    for p in missing:
        print("  -", p)
    raise FileNotFoundError("Copy the v5 scripts into am_flexdc/train and verify repo paths.")

print("All required scripts found.")

## 7. W&B login — RUN if W&B is enabled

In [ ]:
if USE_WANDB and WANDB_MODE != "disabled":
    import os, getpass, wandb
    os.environ.pop("WANDB_BASE_URL", None)
    os.environ["WANDB_MODE"] = WANDB_MODE

    if WANDB_MODE == "online":
        ok = False
        try:
            from google.colab import userdata
            key_from_secret = userdata.get("WANDB_API_KEY")
        except Exception:
            key_from_secret = None
        if key_from_secret:
            os.environ["WANDB_API_KEY"] = key_from_secret
            ok = wandb.login(key=key_from_secret, relogin=True, verify=True)
        else:
            try:
                ok = wandb.login(relogin=False, verify=True)
            except Exception as exc:
                print("W&B login prompt failed or no key configured:", repr(exc))
                api_key = getpass.getpass("Paste W&B API key, hidden input: ")
                os.environ["WANDB_API_KEY"] = api_key
                ok = wandb.login(key=api_key, relogin=True, verify=True)
        if not ok:
            raise RuntimeError("W&B login did not verify. Set USE_WANDB=False to run without W&B.")
        print("W&B login verified.")
    else:
        print("W&B mode is", WANDB_MODE)
else:
    print("W&B disabled.")

## 8. Model and dataset configuration — RUN

In [ ]:
TARGET_FAMILY = "flexdc"
TARGET_MODE = "raw"
RAW_QOS_AGGREGATION = "mean"
USE_NORM_COST = "auto"
USE_NORM_PR = "true"

DATASET_TAG = "newqos_plus_w2dense_v1"
DATASET_DIR = AM_FLEXDC_ROOT / "data" / "pilots" / "traditionaliso_newqos_pilot_plus_w2_dense_v1_flexdc_configured_objective"
RESULTS_CSV = DATASET_DIR / "traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_results.csv"
DIAGNOSTICS_CSV = DATASET_DIR / "traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_diagnostics.csv"
MODEL_FILE = MODELS_DIR / "flexdc_raw" / "am_flexdc_raw_newqos_plus_w2dense_v1_wandb_v2_state_dict.pt"

print("MODEL_FILE:", MODEL_FILE)
print("RESULTS_CSV:", RESULTS_CSV)
print("DIAGNOSTICS_CSV:", DIAGNOSTICS_CSV)
for p in [MODEL_FILE, RESULTS_CSV, DIAGNOSTICS_CSV]:
    print(("OK" if p.exists() else "MISSING"), p)

## 9. Choose test scenario — EDIT/RUN

Weights start as equal by default. The optimizer will still update weights during gradient descent.


In [ ]:
# Presets: "W2_LU", "W2_HU", "W1_LU", "W1_HU", or "CUSTOM".
SCENARIO = "W2_LU"

START_WEIGHTS_MODE = "equal"   # "equal" or "custom"
CUSTOM_START_WEIGHTS = "0.258019617529,0.250894690209,0.252694830182,0.23839086208"

scenario_map = {
    "W2_LU": {
        "workload": FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5_4.5_4_3.5.ini",
        "utilization": 0.60,
        "start_pbar": 0.472128,
        "start_r": 0.102206,
        "server_count": 1000,
        "run_name": "W2_LU_N1000_U060",
    },
    "W2_HU": {
        "workload": FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5555.ini",
        "utilization": 0.80,
        "start_pbar": 0.576789,
        "start_r": 0.138748,
        "server_count": 1000,
        "run_name": "W2_HU_N1000_U080",
    },
    "W1_LU": {
        "workload": FLEXDC_ROOT / "configs" / "workload" / "W1-train-qos3333.ini",
        "utilization": 0.60,
        "start_pbar": 0.528294,
        "start_r": 0.234583,
        "server_count": 1000,
        "run_name": "W1_LU_N1000_U060",
    },
    "W1_HU": {
        "workload": FLEXDC_ROOT / "configs" / "workload" / "W1-train-qos4444.ini",
        "utilization": 0.80,
        "start_pbar": 0.578285,
        "start_r": 0.210078,
        "server_count": 1000,
        "run_name": "W1_HU_N1000_U080",
    },
}

if SCENARIO == "CUSTOM":
    WORKLOAD_CONFIG = FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5_4.5_4_3.5.ini"
    UTILIZATION = 0.60
    START_PBAR = 0.472128
    START_R = 0.102206
    SERVER_COUNT = 1000
    RUN_NAME = "custom"
else:
    s = scenario_map[SCENARIO]
    WORKLOAD_CONFIG = s["workload"]
    UTILIZATION = s["utilization"]
    START_PBAR = s["start_pbar"]
    START_R = s["start_r"]
    SERVER_COUNT = s["server_count"]
    RUN_NAME = s["run_name"]

if START_WEIGHTS_MODE == "equal":
    START_WEIGHTS = "0.25,0.25,0.25,0.25"
elif START_WEIGHTS_MODE == "custom":
    START_WEIGHTS = CUSTOM_START_WEIGHTS
else:
    raise ValueError("START_WEIGHTS_MODE must be equal or custom")

EXPERIMENT_CONFIG = FLEXDC_ROOT / "configs" / "experiment" / "new_iso" / "traditional_signal" / "generated_server_counts" / "exp_traditional_iso16_servers_1000.ini"

print("Scenario:", SCENARIO)
print("Workload:", WORKLOAD_CONFIG)
print("Utilization:", UTILIZATION)
print("Start Pbar/R:", START_PBAR, START_R)
print("Start weights mode:", START_WEIGHTS_MODE)
print("Start weights:", START_WEIGHTS)
print("Experiment config:", EXPERIMENT_CONFIG)

## 10. Optimization controls — EDIT/RUN

`RAW_OBJECTIVE_MODE="paper_approx"` uses the raw model outputs and the paper-form constants. Set `RAW_OBJECTIVE_MODE="weighted_sum"` and `OBJECTIVE_WEIGHTS="auto"` for the equal raw baseline.


In [ ]:
DEVICE = "cuda"   # "cuda", "cpu", or "auto"
ITERATIONS = 1000
LR = 0.001

# For flexdc/raw:
#   paper_approx = M_RSR + psi*softplus(mu*(epsilon90-gamma)) + beta*J*softplus(rho*(qos_mean-delta))
#   weighted_sum = objective_weights dot [M_RSR, epsilon90, qos_mean]
RAW_OBJECTIVE_MODE = "paper_approx"   # "paper_approx" or "weighted_sum"
OBJECTIVE_WEIGHTS = "auto"            # used only if RAW_OBJECTIVE_MODE == "weighted_sum"; auto -> [1,1,1] for FlexDC

# Paper-form constants used in paper_approx optimization and reporting.
PAPER_CTRACK_PSI = 1.0
PAPER_CTRACK_MU = 10.0
PAPER_CTRACK_GAMMA = 0.3
PAPER_QOS_BETA = 20.0
PAPER_QOS_RHO = 2.0
PAPER_QOS_THRESHOLD = 0.1

# Optional focused bounds. Leave false for general broad-search behavior.
USE_FOCUSED_BOUNDS = False
FOCUSED_BOUNDS = {
    "W2_LU": {"pbar_min": 0.464, "pbar_max": 0.480, "r_max": 0.140},
    "W2_HU": {"pbar_min": 0.570, "pbar_max": 0.593, "r_max": 0.180},
}

Pbar_lower_factor = 0.9
Pbar_upper_factor = 1.0
PR_upper_factor = 1.2
R_lower = 0.01

E2E_DIR = RESULTS_DIR / f"e2e_flexdc_raw_{RUN_NAME}_v5_{RAW_OBJECTIVE_MODE}_start_{START_WEIGHTS_MODE}"
print("RAW_OBJECTIVE_MODE:", RAW_OBJECTIVE_MODE)
print("OBJECTIVE_WEIGHTS:", OBJECTIVE_WEIGHTS)
print("USE_FOCUSED_BOUNDS:", USE_FOCUSED_BOUNDS)
print("E2E_DIR:", E2E_DIR)

## 11. Compile scripts — RUN

In [ ]:
%cd {TRAIN_DIR}
!python -m py_compile \
  am_unified_predict_one.py \
  am_unified_optimize_one_raw_v5.py \
  am_unified_end_to_end_eval_raw_report_v5.py
print("Script syntax check complete.")

## 12. Predict one point — OPTIONAL model-only test

In [ ]:
%cd {TRAIN_DIR}
PREDICT_OUT = RESULTS_DIR / f"predict_one_flexdc_raw_{RUN_NAME}.json"
!python am_unified_predict_one.py \
  --model-file "{MODEL_FILE}" \
  --norm-source-results-csv "{RESULTS_CSV}" \
  --workload-config "{WORKLOAD_CONFIG}" \
  --experiment-config "{EXPERIMENT_CONFIG}" \
  --target-family "{TARGET_FAMILY}" \
  --target-mode "{TARGET_MODE}" \
  --raw-qos-aggregation "{RAW_QOS_AGGREGATION}" \
  --use-norm-cost "{USE_NORM_COST}" \
  --use-norm-pr "{USE_NORM_PR}" \
  --server-count "{SERVER_COUNT}" \
  --utilization "{UTILIZATION}" \
  --pbar-kw-per-server "{START_PBAR}" \
  --r-kw-per-server "{START_R}" \
  --weights "{START_WEIGHTS}" \
  --device "{DEVICE}" \
  --out-json "{PREDICT_OUT}"
print("Saved prediction:", PREDICT_OUT)

## 13. Optimize only — OPTIONAL dry run

In [ ]:
import subprocess, shlex, sys
from pathlib import Path

%cd {TRAIN_DIR}
OPT_DIR = RESULTS_DIR / f"optimize_only_flexdc_raw_{RUN_NAME}_v5_{RAW_OBJECTIVE_MODE}_start_{START_WEIGHTS_MODE}"
cmd = [
    sys.executable, "am_unified_optimize_one_raw_v5.py",
    "--model-file", str(MODEL_FILE),
    "--norm-source-results-csv", str(RESULTS_CSV),
    "--workload-config", str(WORKLOAD_CONFIG),
    "--experiment-config", str(EXPERIMENT_CONFIG),
    "--target-family", TARGET_FAMILY,
    "--target-mode", TARGET_MODE,
    "--raw-qos-aggregation", RAW_QOS_AGGREGATION,
    "--use-norm-cost", USE_NORM_COST,
    "--use-norm-pr", USE_NORM_PR,
    "--server-count", str(SERVER_COUNT),
    "--utilization", str(UTILIZATION),
    "--start-pbar-kw-per-server", str(START_PBAR),
    "--start-r-kw-per-server", str(START_R),
    "--start-weights", START_WEIGHTS,
    "--iterations", str(ITERATIONS),
    "--lr", str(LR),
    "--objective-weights", OBJECTIVE_WEIGHTS,
    "--raw-objective-mode", RAW_OBJECTIVE_MODE,
    "--paper-ctrack-psi", str(PAPER_CTRACK_PSI),
    "--paper-ctrack-mu", str(PAPER_CTRACK_MU),
    "--paper-ctrack-gamma", str(PAPER_CTRACK_GAMMA),
    "--paper-qos-beta", str(PAPER_QOS_BETA),
    "--paper-qos-rho", str(PAPER_QOS_RHO),
    "--paper-qos-threshold", str(PAPER_QOS_THRESHOLD),
    "--pbar-lower-factor", str(Pbar_lower_factor),
    "--pbar-upper-factor", str(Pbar_upper_factor),
    "--pr-upper-factor", str(PR_upper_factor),
    "--r-lower-kw-per-server", str(R_lower),
    "--device", DEVICE,
    "--out-dir", str(OPT_DIR),
]
if USE_FOCUSED_BOUNDS and SCENARIO in FOCUSED_BOUNDS:
    b = FOCUSED_BOUNDS[SCENARIO]
    cmd += ["--pbar-min-kw-per-server", str(b["pbar_min"]), "--pbar-max-kw-per-server", str(b["pbar_max"]), "--r-max-kw-per-server", str(b["r_max"])]

print("Running optimize-only command:")
print(" ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)
print("Saved optimize-only outputs:", OPT_DIR)

## 14. Full end-to-end evaluation — RUN when ready

In [ ]:
import subprocess, shlex, sys
from pathlib import Path

%cd {TRAIN_DIR}
E2E_DIR.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, "am_unified_end_to_end_eval_raw_report_v5.py",
    "--model-file", str(MODEL_FILE),
    "--norm-source-results-csv", str(RESULTS_CSV),
    "--workload-config", str(WORKLOAD_CONFIG),
    "--experiment-config", str(EXPERIMENT_CONFIG),
    "--target-family", TARGET_FAMILY,
    "--target-mode", TARGET_MODE,
    "--raw-qos-aggregation", RAW_QOS_AGGREGATION,
    "--use-norm-cost", USE_NORM_COST,
    "--use-norm-pr", USE_NORM_PR,
    "--server-count", str(SERVER_COUNT),
    "--utilization", str(UTILIZATION),
    "--start-pbar-kw-per-server", str(START_PBAR),
    "--start-r-kw-per-server", str(START_R),
    "--start-weights", START_WEIGHTS,
    "--iterations", str(ITERATIONS),
    "--lr", str(LR),
    "--objective-weights", OBJECTIVE_WEIGHTS,
    "--raw-objective-mode", RAW_OBJECTIVE_MODE,
    "--paper-ctrack-psi", str(PAPER_CTRACK_PSI),
    "--paper-ctrack-mu", str(PAPER_CTRACK_MU),
    "--paper-ctrack-gamma", str(PAPER_CTRACK_GAMMA),
    "--paper-qos-beta", str(PAPER_QOS_BETA),
    "--paper-qos-rho", str(PAPER_QOS_RHO),
    "--paper-qos-threshold", str(PAPER_QOS_THRESHOLD),
    "--pbar-lower-factor", str(Pbar_lower_factor),
    "--pbar-upper-factor", str(Pbar_upper_factor),
    "--pr-upper-factor", str(PR_upper_factor),
    "--r-lower-kw-per-server", str(R_lower),
    "--device", DEVICE,
    "--flexdc-root", str(FLEXDC_ROOT),
    "--flexdc-python", sys.executable,
    "--run-flexdc",
    "--out-dir", str(E2E_DIR),
]
if USE_FOCUSED_BOUNDS and SCENARIO in FOCUSED_BOUNDS:
    b = FOCUSED_BOUNDS[SCENARIO]
    cmd += ["--pbar-min-kw-per-server", str(b["pbar_min"]), "--pbar-max-kw-per-server", str(b["pbar_max"]), "--r-max-kw-per-server", str(b["r_max"])]
if USE_WANDB and WANDB_MODE != "disabled":
    cmd += ["--wandb-project", WANDB_PROJECT, "--wandb-entity", WANDB_ENTITY, "--wandb-run-name", f"e2e-flexdc-raw-{RUN_NAME}-v5-{RAW_OBJECTIVE_MODE}-start-{START_WEIGHTS_MODE}", "--wandb-mode", WANDB_MODE]

print("Running end-to-end command:")
print(" ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)
print("Saved E2E outputs:", E2E_DIR)

## 15. Nicely formatted raw predicted-vs-actual table — RUN after E2E

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Markdown

EVAL_OUT_DIR = Path(E2E_DIR)
key_path = EVAL_OUT_DIR / "end_to_end_raw_key_summary.csv"
full_path = EVAL_OUT_DIR / "end_to_end_validation_summary.csv"
if not key_path.exists():
    raise FileNotFoundError(f"Missing key summary: {key_path}. Run full E2E cell first.")

key = pd.read_csv(key_path)
full = pd.read_csv(full_path) if full_path.exists() else key.copy()

MAX_P90 = 0.30
MAX_QOS_RATIO = 0.10

def short_weights(s, digits=4):
    try:
        vals = json.loads(str(s).replace("'", '"'))
    except Exception:
        import ast, re
        txt = re.sub(r"np\.float64\(([^()]*)\)", r"\1", str(s))
        vals = ast.literal_eval(txt)
    return "[" + ", ".join(f"{float(x):.{digits}f}" for x in vals) + "]"

rows = []
for _, r in key.iterrows():
    rows.append({
        "Configuration": r["Configuration"],
        "Pbar": r["Pbar_kw_per_server"],
        "R": r["R_kw_per_server"],
        "Pbar+R": r["Pbar_plus_R"],
        "Pbar-R": r["Pbar_minus_R"],
        "Weights": short_weights(r["Weights"]),
        "Pred M_RSR": r.get("Predicted_flexdc_M_RSR", np.nan),
        "Actual M_RSR": r.get("Actual_flexdc_M_RSR", np.nan),
        "Pred p90": r.get("Predicted_raw_Ctrack_Epsilon_90th", np.nan),
        "Actual p90": r.get("Actual_raw_Ctrack_Epsilon_90th", np.nan),
        "Pred QoS mean": r.get("Predicted_raw_qos_probability_mean", np.nan),
        "Actual QoS mean": r.get("Actual_raw_qos_probability_mean", np.nan),
        "QoS ratio": r.get("QoS_Violation_Ratio", np.nan),
        "Max QoS prob": r.get("Max_QoS_Delay_Probability", np.nan),
        "Tracking pass": bool(r.get("Tracking_Pass", False)),
        "QoS pass": bool(r.get("QoS_Pass_CurrentLogic", False)),
        "Both pass": bool(r.get("Both_Pass_CurrentLogic", False)),
        "Pred paper approx": r.get("Predicted_PaperObjective_Approx", np.nan),
        "Actual paper obj": r.get("paper_objective_from_raw", np.nan),
    })
view = pd.DataFrame(rows)

fmt_cols = [c for c in view.columns if c not in ["Configuration", "Weights", "Tracking pass", "QoS pass", "Both pass"]]
for c in fmt_cols:
    view[c] = pd.to_numeric(view[c], errors="coerce")

# Main interpretation row
selected = view[view["Configuration"].str.contains("Selected", case=False, na=False)].iloc[0]
start = view[view["Configuration"].str.contains("Starting", case=False, na=False)].iloc[0]

if bool(selected["Both pass"]):
    verdict = "Selected configuration passes both constraints."
    verdict_color = "#064e3b"
else:
    verdict = "Selected configuration fails at least one constraint; do not trust it without changing objective/constraints."
    verdict_color = "#7f1d1d"

html = f"""
<div style='background:{verdict_color}; color:white; padding:10px; border-radius:8px; font-weight:800; margin-bottom:10px;'>
{verdict}<br>
Objective mode: <code>{RAW_OBJECTIVE_MODE}</code> | Start weights: <code>{START_WEIGHTS_MODE}</code> | Focused bounds: <code>{USE_FOCUSED_BOUNDS}</code>
</div>
"""
display(HTML(html))

def style_pass(v):
    if isinstance(v, (bool, np.bool_)):
        return "background-color: #064e3b; color: #ecfdf5; font-weight: 800" if v else "background-color: #7f1d1d; color: #fef2f2; font-weight: 800"
    return ""

def style_numeric_column(col):
    styles = []
    for idx, val in col.items():
        s = ""
        if col.name == "Actual p90" and pd.notna(val):
            s = "background-color: #064e3b; color: white" if float(val) <= MAX_P90 else "background-color: #7f1d1d; color: white"
        if col.name == "QoS ratio" and pd.notna(val):
            s = "background-color: #064e3b; color: white" if float(val) <= MAX_QOS_RATIO else "background-color: #7f1d1d; color: white"
        styles.append(s)
    return styles

styled = (
    view.style
    .format({c: "{:.6f}" for c in fmt_cols})
    .hide(axis="index")
    .map(style_pass, subset=["Tracking pass", "QoS pass", "Both pass"])
    .apply(style_numeric_column, subset=["Actual p90", "QoS ratio"])
    .set_properties(**{"text-align":"left", "white-space":"normal", "font-size":"12px", "border":"1px solid #cbd5e1", "padding":"6px"})
    .set_table_styles([
        {"selector":"th", "props":[("background-color", "#0f172a"), ("color","white"), ("font-weight","800"), ("text-align","left"), ("padding","7px")]},
        {"selector":"table", "props":[("border-collapse","collapse"), ("width","100%")]}])
)
display(Markdown("### Raw FlexDC predicted vs actual key table"))
display(styled)
print("Source:", key_path)

# Smaller P/R/W table for quick screenshotting.
prw = view[["Configuration", "Pbar", "R", "Pbar+R", "Pbar-R", "Weights", "Both pass"]].copy()
display(Markdown("### P/R/weights summary"))
display(prw.style.format({"Pbar":"{:.6f}", "R":"{:.6f}", "Pbar+R":"{:.6f}", "Pbar-R":"{:.6f}"}).hide(axis="index").map(style_pass, subset=["Both pass"]))

## 16. Paper-form constants playground — RUN after E2E

In [ ]:
from pathlib import Path
import json, ast, re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Edit constants here to inspect sensitivity. These do not rerun optimization.
PSI = PAPER_CTRACK_PSI
MU = PAPER_CTRACK_MU
GAMMA = PAPER_CTRACK_GAMMA
BETA = PAPER_QOS_BETA
RHO = PAPER_QOS_RHO
DELTA = PAPER_QOS_THRESHOLD

key_path = Path(E2E_DIR) / "end_to_end_raw_key_summary.csv"
key = pd.read_csv(key_path)

def softplus(x):
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0)

def parse_probs(x):
    try:
        return [float(v) for v in json.loads(str(x))]
    except Exception:
        txt = re.sub(r"np\.float64\(([^()]*)\)", r"\1", str(x))
        return [float(v) for v in ast.literal_eval(txt)]

rows=[]
for _, r in key.iterrows():
    probs = parse_probs(r.get("QoS_Delay_Probabilities", "[]")) if "QoS_Delay_Probabilities" in r.index else []
    eps = float(r.get("Actual_raw_Ctrack_Epsilon_90th", np.nan))
    m = float(r.get("Actual_flexdc_M_RSR", np.nan))
    actual_ctrack = float(PSI * softplus(MU * (eps - GAMMA)))
    actual_cqos = float(BETA * np.sum(softplus(RHO * (np.asarray(probs, dtype=float) - DELTA)))) if probs else np.nan
    actual_obj = m + actual_ctrack + actual_cqos

    pred_eps = float(r.get("Predicted_raw_Ctrack_Epsilon_90th", np.nan))
    pred_qmean = float(r.get("Predicted_raw_qos_probability_mean", np.nan))
    pred_m = float(r.get("Predicted_flexdc_M_RSR", np.nan))
    pred_ctrack = float(PSI * softplus(MU * (pred_eps - GAMMA)))
    # Approximate because model predicts QoS mean, not per-job vector.
    pred_cqos = float(BETA * 4 * softplus(RHO * (pred_qmean - DELTA)))
    pred_obj = pred_m + pred_ctrack + pred_cqos

    rows.append({
        "Configuration": r["Configuration"],
        "Pbar": r["Pbar_kw_per_server"],
        "R": r["R_kw_per_server"],
        "Actual_M_RSR": m,
        "Actual_Ctrack_const": actual_ctrack,
        "Actual_CQoS_const": actual_cqos,
        "Actual_PaperObjective_const": actual_obj,
        "Pred_Ctrack_approx": pred_ctrack,
        "Pred_CQoS_approx": pred_cqos,
        "Pred_PaperObjective_approx": pred_obj,
        "QoS_probs": probs,
    })

out = pd.DataFrame(rows)
display(Markdown(f"### Paper-form objective with psi={PSI}, mu={MU}, gamma={GAMMA}, beta={BETA}, rho={RHO}, delta={DELTA}"))
display(out.style.format({c:"{:.6f}" for c in out.columns if c not in ["Configuration", "QoS_probs"]}).hide(axis="index"))
print("Note: predicted QoS cost is approximate because the current raw model predicts mean QoS probability, not the per-job-type vector.")

## 17. Inspect and download outputs

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

print("Evaluation folder:", E2E_DIR)
for p in sorted(Path(E2E_DIR).glob("*")):
    print(" -", p.name)

try:
    from google.colab import files
    for name in ["end_to_end_raw_key_summary.csv", "end_to_end_validation_summary.csv", "end_to_end_validation_report.md", "optimization_trajectory.csv", "optimized_candidate.json"]:
        p = Path(E2E_DIR) / name
        if p.exists():
            print("Available for download:", p)
            # Uncomment to download one at a time:
            # files.download(str(p))
except Exception:
    print("Not in Colab or google.colab.files unavailable.")